# Spoken dialogue, timed: where the response latency goes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/sds_latency_demo.ipynb) [![sds_latency_demo](https://github.com/espnet/notebook/actions/workflows/sds_latency_demo.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/sds_latency_demo.yml)

A cascaded spoken dialogue system is four models in series: voice activity
detection (VAD) decides that you have stopped, speech recognition (ASR) writes
down what you said, a language model (LLM) answers, and speech synthesis (TTS)
says it. Nothing is heard until the last one has produced sound, so their
delays add. This notebook times each stage on one real turn, adds them up,
tries one way to start speaking sooner, and then lets you talk to it.

Every stage is a class of `espnet2.sds`, the same as in the short
[`sds_demo.ipynb`](sds_demo.ipynb), with the same models.

| stage | class | model |
|---|---|---|
| VAD | `FrontiaVADModel` | `espnet/frontia_vad_v2_3` |
| ASR | `OWSMCTCModel` | `espnet/owsm_ctc_v4_1B` |
| LLM | `HuggingFaceLLM` | `Qwen/Qwen2.5-0.5B-Instruct` |
| TTS | `ESPnetTTSModel` | `espnet/kan-bayashi_ljspeech_vits` |

In Colab, **Runtime → Change runtime type → T4 GPU** gives the numbers a
dialogue system would see; on a CPU every model stage takes seconds, and the
point of the exercise shows all the more.

## Install

In [ ]:
%pip install -q "espnet[tts]==202611.post1" espnet_model_zoo librosa transformers matplotlib

## Load the four models once

Loaded and warmed up before anything is timed, so that the numbers below are
inference, not download or first-call setup.

In [ ]:
import time

import numpy as np
import torch

from espnet2.sds.asr.owsm_ctc_asr import OWSMCTCModel
from espnet2.sds.llm.hugging_face_llm import HuggingFaceLLM
from espnet2.sds.tts.espnet_tts import ESPnetTTSModel
from espnet2.sds.vad.frontia_vad import FrontiaVADModel

DEV = "cuda" if torch.cuda.is_available() else "cpu"
FS = 16000
load = {}

t0 = time.time()
vad = FrontiaVADModel(tag="espnet/frontia_vad_v2_3", device=DEV)
vad.warmup()
load["VAD"] = time.time() - t0

t0 = time.time()
asr = OWSMCTCModel(tag="espnet/owsm_ctc_v4_1B", device=DEV)
asr.warmup()
load["ASR"] = time.time() - t0

t0 = time.time()
llm = HuggingFaceLLM(tag="Qwen/Qwen2.5-0.5B-Instruct", device=DEV,
                     dtype="float16" if DEV == "cuda" else "float32",
                     max_new_tokens=40)
llm.warmup()
load["LLM"] = time.time() - t0

t0 = time.time()
tts = ESPnetTTSModel(tag="espnet/kan-bayashi_ljspeech_vits", device=DEV)
tts.warmup()
load["TTS"] = time.time() - t0

print(f"device: {DEV}")
for k, v in load.items():
    print(f"  {k} loaded in {v:5.1f} s")

## The user's turn

A real recording, with half a second of silence before it and two seconds
after, as a microphone would give it: the VAD can only say the turn is over
once it has heard the silence that follows.

In [ ]:
import librosa
from IPython.display import Audio, display

!wget -q -O turn.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
speech, _ = librosa.load("turn.wav", sr=FS)
user = np.concatenate([np.zeros(FS // 2, np.float32), speech,
                       np.zeros(2 * FS, np.float32)])
print(f"{len(speech) / FS:.2f} s of speech, {len(user) / FS:.2f} s in all")
display(Audio(user, rate=FS))

## Stage 1: when has the user stopped?

The VAD is fed 20 ms at a time, as a microphone would feed it, and returns
the turn once speech has been followed by `hangover_ms` (500 ms) of silence.
It gives every 8 ms frame a speech probability, thresholds it, and smooths
the 0/1 sequence with a **centred** median filter of 25 frames.

The model reads no future audio, but the default decision is **not strictly
causal**: the centred median waits for the 12 frames after each frame. Two
delays, before any compute:

* a frame's probability needs its whole 512-sample window: **32 ms**;
* the centred median then needs 12 more frames: **96 ms**
  (`median=1` removes it, and the decision becomes strictly causal).

So the turn ends `endpoint_delay_s` after the speech does: those 128 ms plus
the 500 ms of silence it waits for.

In [ ]:
import matplotlib.pyplot as plt

vad.reset()
chunk = FS // 50
compute, chunks = 0.0, 0
turn, returned_at = None, None
for i in range(0, len(user), chunk):
    t0 = time.time()
    turn = vad.forward(user[i:i + chunk], FS)
    compute += time.time() - t0
    chunks += 1
    if turn is not None:
        returned_at = (i + chunk) / FS
        break
info = vad.last_turn
T = {"VAD": returned_at - info["endpoint_s"]}
print(f"speech from {info['speech_start_s']:.2f} s to {info['endpoint_s']:.2f} s")
print(f"turn returned after {returned_at:.2f} s of audio: "
      f"{T['VAD'] * 1000:.0f} ms after the speech ended "
      f"(endpoint delay {vad.endpoint_delay_s * 1000:.0f} ms by construction)")
print(f"compute: {compute / chunks * 1000:.1f} ms per 20 ms chunk")

t = info["probs_start_s"] + np.arange(len(info["probs"])) * info["frame_shift"]
fig, ax = plt.subplots(figsize=(10, 2.5))
ax.plot(t, info["probs"], lw=1, label="P(speech)")
ax.plot(t[:len(info["decisions"])], info["decisions"], lw=1.5,
        label="after the centred median")
ax.axvline(info["endpoint_s"], color="r", lw=1, label="end of speech")
ax.axvline(returned_at, color="k", ls="--", lw=1, label="turn returned")
ax.set_xlabel("time (s)"); ax.legend(loc="lower left", fontsize=8)
plt.tight_layout(); plt.show()

## Stage 2: what did they say?

`OWSMCTCModel`, greedy CTC decoding in one encoder pass over a 30 s window.
The turn the VAD returned already carries 300 ms of audio before the speech
and after it, so nothing is clipped and nothing more is waited for.

In [ ]:
t0 = time.time()
heard = asr.forward(turn)
T["ASR"] = time.time() - t0
print(f"{T['ASR'] * 1000:.0f} ms: {heard!r}")

## Stage 3: the language model answers

`HuggingFaceLLM.stream` yields the reply as it is generated, so two times can
be read: when the first text arrives, and when the reply is complete. The
cascade as written waits for the second.

In [ ]:
messages = [{"role": "system", "content": "You are a helpful assistant in a spoken conversation. Reply to what the user just said in one short, natural sentence."},
            {"role": "user", "content": heard}]
t0 = time.time()
first, reply = None, ""
for piece in llm.stream(messages):
    if first is None and piece.strip():
        first = time.time() - t0
    reply += piece
reply = reply.strip()
T["LLM"] = time.time() - t0
print(f"first text after {first * 1000:.0f} ms, "
      f"whole reply after {T['LLM'] * 1000:.0f} ms")
print(reply)

## Stage 4: speech synthesis

VITS is a single stage, with no separate vocoder, so the first sample of
speech is ready when the whole forward pass is.

In [ ]:
t0 = time.time()
fs_out, wav = tts.forward(reply)
T["TTS"] = time.time() - t0
dur = len(wav) / fs_out
print(f"{T['TTS'] * 1000:.0f} ms to synthesise {dur:.2f} s of speech "
      f"(real-time factor {T['TTS'] / dur:.3f})")
display(Audio(wav, rate=fs_out))

## The response latency, stage by stage

The column that matters to the person waiting is the cumulative one: nothing
is heard until the last row. Compare it with **230 ms**, the average gap
between turns in human conversation (Stivers et al., 2009).

In [ ]:
rows = [("VAD: end of speech to end of turn", T["VAD"]),
        ("ASR", T["ASR"]),
        ("LLM: whole reply", T["LLM"]),
        ("TTS: to the first sample", T["TTS"])]
total = sum(v for _, v in rows)
print(f"{'stage':36s}{'ms':>8}{'cumulative':>12}{'share':>8}")
cum = 0.0
for name, v in rows:
    cum += v
    print(f"{name:36s}{v * 1000:8.0f}{cum * 1000:12.0f}{v / total * 100:7.0f}%")
print(f"{'response latency':36s}{total * 1000:8.0f}")

## Speaking before the reply is finished

The cascade waits for the whole reply before it synthesises anything. Instead,
synthesise the first clause as soon as the language model has written it: the
first speech arrives sooner, at the cost of choosing that clause's prosody
without knowing how the sentence ends.

In [ ]:
import re

# One generation, two times read from it: when the first clause is complete,
# and when the whole reply is. Two separate runs would also measure how much
# the LLM's speed varies from run to run.
t0 = time.time()
text, clause, t_clause = "", None, None
for piece in llm.stream(messages):
    text += piece
    if clause is None:
        m = re.search(r"^(.+?[,.;:!?])(\s|$)", text.strip())
        if m:
            clause, t_clause = m.group(1), time.time() - t0
t_full = time.time() - t0
full = text.strip()
if clause is None or clause == full:
    print(f"the reply is one clause, so nothing can start early: {full!r}")
else:
    t0 = time.time(); _, first_wav = tts.forward(clause); tts_clause = time.time() - t0
    t0 = time.time(); tts.forward(full); tts_full = time.time() - t0
    whole = t_full + tts_full
    early = t_clause + tts_clause
    print(f"whole reply, then TTS : {whole * 1000:6.0f} ms to the first speech")
    print(f"first clause, then TTS: {early * 1000:6.0f} ms to the first speech   {clause!r}")
    print(f"saved {(whole - early) * 1000:.0f} ms")
    display(Audio(first_wav, rate=fs_out))

## Talk to it

**Colab only**: this cell records from your browser's microphone. Speak, then
pause; the browser stops recording after `SILENCE_MS` of quiet, the VAD finds
where you stopped, and the system answers, then listens again. Each turn
prints its own latency, and the end of the loop prints the median and the
90th percentile over the conversation, which is how a dialogue system's
latency is reported: one turn is not a measurement.

Say "goodbye" to stop. Outside Colab the cell only says it was skipped.

In [ ]:
import base64
import re
import subprocess

import soundfile as sf
from IPython.display import HTML

try:
    from google.colab import output as colab_output
except ImportError:
    colab_output = None

MAX_TURNS = 8
SILENCE_MS = 800       # quiet the browser waits for before it stops recording
MAX_TURN_MS = 15000    # longest turn it records
RMS_THRESH = 0.02      # browser-side speech threshold
KEEP_TURNS = 4         # exchanges kept in the prompt

LISTEN_JS = r"""
(async (maxMs, silenceMs, thresh) => {
  const box = document.createElement('div');
  box.style.cssText = 'font:14px monospace;padding:6px 0;white-space:pre';
  document.body.appendChild(box);
  try {
    const s = await navigator.mediaDevices.getUserMedia({audio: true});
    const ctx = new AudioContext();
    const an = ctx.createAnalyser(); an.fftSize = 1024;
    ctx.createMediaStreamSource(s).connect(an);
    const buf = new Float32Array(an.fftSize);
    const rec = new MediaRecorder(s), chunks = [];
    rec.ondataavailable = e => chunks.push(e.data);
    rec.start();
    const t0 = Date.now(); let lastVoice = Date.now(), started = false;
    while (true) {
      await new Promise(r => setTimeout(r, 50));
      an.getFloatTimeDomainData(buf);
      let rms = 0; for (const v of buf) rms += v * v;
      rms = Math.sqrt(rms / buf.length);
      if (rms > thresh) { lastVoice = Date.now(); started = true; }
      const bars = Math.min(40, Math.round(rms * 500));
      box.textContent = (started ? 'REC ' : '    ') + '[' + '#'.repeat(bars)
        + '.'.repeat(40 - bars) + '] ' + (started ? 'speak, then pause' : 'speak now');
      if (started && Date.now() - lastVoice > silenceMs) break;
      if (Date.now() - t0 > maxMs) break;
    }
    rec.stop(); await new Promise(r => rec.onstop = r);
    s.getTracks().forEach(t => t.stop()); ctx.close();
    // FileReader, not btoa(String.fromCharCode(...bytes)): spreading a long
    // recording into arguments overflows the call stack.
    return await new Promise(res => {
      const fr = new FileReader();
      fr.onloadend = () => res(fr.result.split(',')[1]);
      fr.readAsDataURL(new Blob(chunks));
    });
  } finally { box.remove(); }
})(MAXMS, SILMS, THRESH)
"""


def listen():
    js = (LISTEN_JS.replace("MAXMS", str(MAX_TURN_MS))
          .replace("SILMS", str(SILENCE_MS)).replace("THRESH", str(RMS_THRESH)))
    open("mic.webm", "wb").write(base64.b64decode(colab_output.eval_js(js)))
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "mic.webm",
                    "-ar", str(FS), "-ac", "1", "mic.wav"], check=True)
    audio = sf.read("mic.wav", dtype="float32")[0]
    # the browser stops after SILENCE_MS; pad so the VAD hears its hangover too
    return np.concatenate([audio, np.zeros(FS, np.float32)])


if colab_output is None:
    print("No Colab microphone here: this cell is skipped.")
else:
    history = [{"role": "system", "content": "You are a helpful assistant in a spoken conversation. Reply to what the user just said in one short, natural sentence."}]
    latencies = []
    for n in range(1, MAX_TURNS + 1):
        display(HTML(f"<b>turn {n}</b>: speak now"))
        audio = listen()
        vad.reset()
        t = {}
        turn = None
        for i in range(0, len(audio), FS // 50):
            turn = vad.forward(audio[i:i + FS // 50], FS)
            if turn is not None:
                break
        if turn is None:
            print("no speech heard; listening again")
            continue
        t["vad"] = vad.endpoint_delay_s
        t0 = time.time(); said = asr.forward(turn); t["asr"] = time.time() - t0
        print("you:", said)
        history.append({"role": "user", "content": said})
        prompt = history[:1] + history[-(2 * KEEP_TURNS + 1):]
        t0 = time.time(); answer = llm.forward(prompt); t["llm"] = time.time() - t0
        history.append({"role": "assistant", "content": answer})
        print("bot:", answer)
        t0 = time.time(); fs_out, wav = tts.forward(answer); t["tts"] = time.time() - t0
        display(Audio(wav, rate=fs_out, autoplay=True))
        total = sum(t.values())
        latencies.append(total)
        print("  " + " + ".join(f"{k} {v * 1000:.0f}" for k, v in t.items())
              + f" = {total * 1000:.0f} ms")
        time.sleep(len(wav) / fs_out)
        if re.search(r"\b(goodbye|bye|stop|quit)\b", said, re.I):
            break
    if latencies:
        lat = np.array(latencies) * 1000
        print(f"{len(lat)} turns: median {np.median(lat):.0f} ms, "
              f"90th percentile {np.percentile(lat, 90):.0f} ms")

## Where next

- **The short version**, one call per stage: [`sds_demo.ipynb`](sds_demo.ipynb)
- **The app**: [`egs2/TEMPLATE/sds1`](https://github.com/espnet/espnet/tree/master/egs2/TEMPLATE/sds1),
  a Gradio app with an end-to-end option beside the cascade, and latency and
  quality measured while you talk
- **Swap a stage**: every `espnet2.sds` class takes a model tag, so a larger
  LLM (`Qwen/Qwen3-1.7B` fits a T4) or another TTS voice is one line; the
  latency table shows what it costs
- **The VAD on its own**: `espnet detect --live` and the
  [frontia-vad model card](https://huggingface.co/espnet/frontia_vad_v2_3)